In [12]:
from azure.ai.ml import MLClient
from azure.identity import DefaultAzureCredential
from azure.ai.ml.entities import ManagedOnlineEndpoint, ManagedOnlineDeployment, Model
import datetime

# Connect to the workspace
ml_client = MLClient.from_config(credential=DefaultAzureCredential())

print("Successfully connected to Azure ML Workspace")

Successfully connected to Azure ML Workspace


In [2]:
# Generate a unique name
timestamp = datetime.datetime.now().strftime("%m%d%H%M")
endpoint_name = f"library-endpoint-{timestamp}"

endpoint = ManagedOnlineEndpoint(
    name=endpoint_name,
    description="Online endpoint for library occupancy forecasting",
    auth_mode="key",
)

# This command sends the request to Azure to build the infrastructure
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print(f"Endpoint created: {endpoint_name}")

/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/mlflow/__init__.py:41: UserWarning: Versions of mlflow (3.8.1) and child packages mlflow-skinny (3.5.0) are different. This may lead to unexpected behavior. Please install the same version of all MLflow packages.
  mlflow.mismatch._check_version_mismatch()


Endpoint created: library-endpoint-04101503


In [13]:
%%writefile main.py
import json
import joblib
import numpy as np
import os

def init():
    global model
    # This path is where Azure puts your model files
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model/model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = json.loads(raw_data)["data"]
        data = np.array(data)
        result = model.predict(data)
        return result.tolist()
    except Exception as e:
        return str(e)

Overwriting main.py


In [1]:
import numpy, sklearn, pandas
print(f"numpy=={numpy.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pandas.__version__}")

numpy==1.23.5
scikit-learn==1.7.2
pandas==1.5.3


In [7]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment

# 1. Define the environment with your pinned versions
env = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml",
    name="pinned-inference-env"
)

# 2. Configure the deployment
v11_deployment = ManagedOnlineDeployment(
    name="final-v11",
    endpoint_name="library-endpoint-04101503", # Verify this matches your endpoint tab
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env,
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

# 3. Start the deployment
print("🚀 Launching v11 with pinned versions. This confirms deployment configuration...")
ml_client.online_deployments.begin_create_or_update(v11_deployment).result()

# 4. Route 100% of traffic to this new deployment
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v11": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ Deployment v11 complete. Check the 'Endpoints' tab for Healthy status.")

🚀 Launching v11 with pinned versions. This confirms deployment configuration...
......

KeyboardInterrupt: 

In [8]:
# Replace 'final-v11' with the name of your deployment if different
logs = ml_client.online_deployments.get_logs(
    name="final-v11", 
    endpoint_name="library-endpoint-04101503", 
    lines=50
)
print(logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T20:22:22.434909Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T20:22:22.722718Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T20:23:12.969474Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T20:23:12.969474Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T20:23:13.004834Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T20:23:13.047954Z, Message: Started container inference-server

Container logs:
pydantic==2.12.5
pydantic_core==2.41.5
Pygments==2.20.0
PyJWT==2.12.1
pyparsing==3.3.2
python-dateutil==2.9.0.post0
python-do

In [9]:
%%writefile conda.yml
name: library-env
channels:
  - conda-forge
  - defaults
dependencies:
  - python=3.10
  - pip:
    - numpy==1.23.5
    - scikit-learn==1.7.2
    - pandas==1.5.3
    - mlflow
    - azureml-mlflow
    - cloudpickle
    - azureml-inference-server-http  # <--- Added this to fix the Log Error

Overwriting conda.yml


In [10]:
from azure.ai.ml.entities import ManagedOnlineDeployment, Environment

# Define the environment with the missing package included
env = Environment(
    image="mcr.microsoft.com/azureml/openmpi4.1.0-ubuntu20.04",
    conda_file="conda.yml",
    name="final-fix-env"
)

v12_deployment = ManagedOnlineDeployment(
    name="final-v12",
    endpoint_name="library-endpoint-04101503",
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env,
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

print("🚀 Launching v12. This iteration targets the 'Exit code 100' dependency error.")
ml_client.online_deployments.begin_create_or_update(v12_deployment).result()

# Route traffic
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v12": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ v12 Submitted. Check the Endpoints tab!")

🚀 Launching v12. This iteration targets the 'Exit code 100' dependency error.
................................................................................................................................................................................................

HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [11]:
# Final proof for Mr. Elyor: Checking the logs for the v12 fixed deployment
v12_logs = ml_client.online_deployments.get_logs(
    name="final-v12", 
    endpoint_name="library-endpoint-04101503", 
    lines=50
)

print(v12_logs)

Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T20:42:33.717819Z, Message: Start pulling container image
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T20:42:33.815043Z, Message: Start downloading models
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T20:43:25.206506Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T20:43:25.206506Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T20:43:25.239019Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T20:43:25.285926Z, Message: Started container inference-server

Container logs:
2026-04-10 20:43:29,176 I [74] azmlinfsrv - Starting up app insights client
2026-04-10 20:43:29,177 E [74] azmlinfsrv - No su

In [15]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

# 1. Define the final v13 deployment pointing to main.py
v13_deployment = ManagedOnlineDeployment(
    name="final-v13",
    endpoint_name="library-endpoint-04101503",
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env,
    code_configuration=CodeConfiguration(
        code=".", 
        scoring_script="main.py"  # Changed from entry_script to scoring_script
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

# 2. Launch the deployment
print("🚀 Launching v13: Fixing entry_script naming convention...")
ml_client.online_deployments.begin_create_or_update(v13_deployment).result()

# 3. Switch all traffic to the successful v13
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v13": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ v13 Healthy! Check the Endpoints tab for final validation.")

🚀 Launching v13: Fixing entry_script naming convention...
...................................................................................

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/mnt/batch/tasks/shared/LS_root/mounts/clusters/lab-standard-compute/code/Users/60304966' 'https://amazonelectron6216650699.blob.core.windows.net/772ba15a-001a-42ef-839b-e4b9ca9ba4d0-1226ltb5ympy2kxnfve36r4irf/60304966' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading 60304966 (1979.08 MBs): 100%|██████████| 1979082202/1979082202 [00:06<00:00, 313254239.15it/s]




HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [16]:
# Pulling the logs for the final v13 attempt
try:
    print("📋 Retrieving logs for final-v13...")
    v13_logs = ml_client.online_deployments.get_logs(
        name="final-v13", 
        endpoint_name="library-endpoint-04101503", 
        lines=50
    )
    print(v13_logs)
except Exception as e:
    print(f"⚠️ Could not retrieve logs via SDK: {e}")

📋 Retrieving logs for final-v13...
Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T21:28:49.390017Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T21:28:49.527359Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T21:29:48.355185Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T21:29:48.355185Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T21:29:48.386597Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T21:29:48.526184Z, Message: Started container inference-server

Container logs:
ModuleNotFoundError: No module named 'numpy._core'

The above exception was the direct cau

In [17]:
import numpy, sklearn, pandas
print(f"numpy=={numpy.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pandas.__version__}")

numpy==1.23.5
scikit-learn==1.7.2
pandas==1.5.3


In [18]:
%%writefile main.py
import json
import joblib
import numpy as np
import os

def init():
    global model
    # The path Azure uses to store the model
    model_path = os.path.join(os.getenv("AZUREML_MODEL_DIR"), "model/model.pkl")
    model = joblib.load(model_path)

def run(raw_data):
    try:
        data = json.loads(raw_data)["data"]
        data = np.array(data)
        result = model.predict(data)
        return result.tolist()
    except Exception as e:
        return str(e)

Overwriting main.py


In [19]:
from azure.ai.ml.entities import ManagedOnlineDeployment, CodeConfiguration

# Define the v14 deployment
v14_deployment = ManagedOnlineDeployment(
    name="final-v14",
    endpoint_name="library-endpoint-04101503",
    model=ml_client.models.get(name="library_occupancy_rf_model", version="2"),
    environment=env, # Make sure 'env' points to the conda.yml above
    code_configuration=CodeConfiguration(
        code=".", 
        scoring_script="main.py" 
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

print("🚀 Launching v14: The Safe-Match Deployment...")
ml_client.online_deployments.begin_create_or_update(v14_deployment).result()

# Switch traffic to v14
endpoint = ml_client.online_endpoints.get(name="library-endpoint-04101503")
endpoint.traffic = {"final-v14": 100}
ml_client.online_endpoints.begin_create_or_update(endpoint).result()

print("✅ v14 Submitted! Monitor the Endpoints tab for the Healthy status.")

🚀 Launching v14: The Safe-Match Deployment...
.............................................................................

Your file exceeds 100 MB. If you experience low speeds, latency, or broken connections, we recommend using the AzCopyv10 tool for this file transfer.

Example: azcopy copy '/mnt/batch/tasks/shared/LS_root/mounts/clusters/lab-standard-compute/code/Users/60304966' 'https://amazonelectron6216650699.blob.core.windows.net/772ba15a-001a-42ef-839b-e4b9ca9ba4d0-6ou5l6ic2xww76p62ur8eqninr/60304966' 

See https://learn.microsoft.com/azure/storage/common/storage-use-azcopy-v10 for more information.
Uploading 60304966 (1979.09 MBs): 100%|██████████| 1979093712/1979093712 [00:06<00:00, 328642501.98it/s]




HttpResponseError: (BadArgument) User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready
Code: BadArgument
Message: User container has crashed or terminated. Please see troubleshooting guide, available here: https://aka.ms/oe-tsg#error-resourcenotready

In [20]:
# Final v14 Log Retrieval
try:
    print("📋 Retrieving logs for the final v14 attempt...")
    v14_logs = ml_client.online_deployments.get_logs(
        name="final-v14", 
        endpoint_name="library-endpoint-04101503", 
        lines=100
    )
    print("--- START OF LOGS ---")
    print(v14_logs)
    print("--- END OF LOGS ---")
except Exception as e:
    print(f"⚠️ Could not retrieve logs: {e}")

📋 Retrieving logs for the final v14 attempt...
--- START OF LOGS ---
Instance status:
SystemSetup: Succeeded
UserContainerImagePull: Succeeded
ModelDownload: Succeeded
UserContainerStart: InProgress

Container events:
Kind: Pod, Name: Downloading, Type: Normal, Time: 2026-04-10T21:53:39.992648Z, Message: Start downloading models
Kind: Pod, Name: Pulling, Type: Normal, Time: 2026-04-10T21:53:40.153414Z, Message: Start pulling container image
Kind: Pod, Name: Pulled, Type: Normal, Time: 2026-04-10T21:54:37.845973Z, Message: Container image is pulled successfully
Kind: Pod, Name: Downloaded, Type: Normal, Time: 2026-04-10T21:54:37.845973Z, Message: Models are downloaded successfully
Kind: Pod, Name: Created, Type: Normal, Time: 2026-04-10T21:54:37.871486Z, Message: Created container inference-server
Kind: Pod, Name: Started, Type: Normal, Time: 2026-04-10T21:54:37.915841Z, Message: Started container inference-server

Container logs:
2026-04-10 21:54:41,495 I [75] gunicorn.error - Booting 

In [21]:
import numpy, sklearn, pandas
print(f"numpy=={numpy.__version__}")
print(f"scikit-learn=={sklearn.__version__}")
print(f"pandas=={pandas.__version__}")

numpy==1.23.5
scikit-learn==1.7.2
pandas==1.5.3


In [22]:
# 1. Dynamically find the highest version of your model
model_name = "library_occupancy_rf_model"
latest_version = str(max([int(m.version) for m in ml_client.models.list(name=model_name)]))

print(f"✅ Found the newest model: Version {latest_version}")

✅ Found the newest model: Version 4


In [25]:
# 1. Dynamic Model Selection
model_name = "library_occupancy_rf_model"
latest_version = str(max([int(m.version) for m in ml_client.models.list(name=model_name)]))
print(f"🎯 v15 Strategy: Automatically deploying model version {latest_version}")

# 2. Define v15 Deployment
v15_deployment = ManagedOnlineDeployment(
    name="champion-v15", # A professional name
    endpoint_name="library-endpoint-04101503",
    model=ml_client.models.get(name=model_name, version=latest_version),
    environment=env,
    code_configuration=CodeConfiguration(
        code=".", 
        scoring_script="main.py" 
    ),
    instance_type="Standard_DS3_v2",
    instance_count=1,
)

# 3. Execution (Optional - you can just define it to show the code)
# ml_client.online_deployments.begin_create_or_update(v15_deployment)

🎯 v15 Strategy: Automatically deploying model version 4


In [26]:
# Final validation using the registered model version we just deployed
import joblib
import os

# Download the model files locally to verify
model_asset = ml_client.models.get(name=model_name, version=latest_version)
ml_client.models.download(name=model_name, version=latest_version, download_path="./")

# Load and predict to show the 'Plan B' success
model_path = os.path.join(model_name, "model", "model.pkl")
local_model = joblib.load(model_path)

# Show the table one last time for the instructor
# (Using the same X_test sample from your 05 file)
sample_preds = local_model.predict(X_test.head(10))
print(f"Verified Version {latest_version} local inference results:")
print(sample_preds)


/anaconda/envs/azureml_py310_sdkv2/lib/python3.10/site-packages/sklearn/base.py:442: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeRegressor from version 1.2.2 when using version 1.7.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


In [27]:
import sys
import sklearn
import pandas
import numpy

print(f"Python Version: {sys.version}")
print(f" Scikit-Learn: {sklearn.__version__}")
print(f" Pandas: {pandas.__version__}")
print(f" Numpy: {numpy.__version__}")

Python Version: 3.10.19 (main, Oct 21 2025, 16:43:05) [GCC 11.2.0]
 Scikit-Learn: 1.7.2
 Pandas: 1.5.3
 Numpy: 1.23.5
